In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from itertools import islice

/Users/vadimbatalev/Documents/programming/hacatons/tenderhack/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
mixtures_train = pd.read_csv('../data/daimler_mixtures_train.csv')
mixtures_test = pd.read_csv('../data/daimler_mixtures_test.csv')
properties = pd.read_csv('../data/daimler_component_properties.csv')

In [9]:
properties

,Компонент,Наименование партии,Наименование показателя,Единица измерения_по_партиям,Значение показателя
0,Детергент_4,0338.22,"Кинематическая вязкость, при 100°C, ASTM D445",мм²/с,16.68
1,Детергент_4,0338.22,"Массовая доля кальция, ASTM D6481",% масс,6.144
2,Детергент_4,0338.22,"Массовая доля цинка, ASTM D6481",% масс,0.0064
3,Детергент_4,0338.22,"Щелочное число, ASTM D2896",мг KOH/г,168
4,Противоизносная_присадка_8,0186.22,"Кинематическая вязкость, при 100°C, ASTM D445",мм²/с,13.74
...,...,...,...,...,...
2636,Базовое_масло_8,typical,"Содержание серы, мг/кг",NaN,<10
2637,Базовое_масло_4,typical,"Содержание серы, мг/кг",NaN,<10
2638,Базовое_масло_16,typical,Содержание ароматики,NaN,<3
2639,Базовое_масло_14,typical,Содержание ароматики,NaN,<3


- Антипенная присадка - все показатели для всех свойств либо NaN, либо 0 - выкидываем из словаря

In [10]:
import re

properties['component_group'] = properties['Компонент'].apply(lambda x: re.sub(r'_\d+$', '', x))
properties['Значение показателя'] = properties['Значение показателя'].astype(str).str.replace('<', '', regex=False).str.replace('>', '', regex=False).str.replace('≤', '', regex=False)
properties['Значение показателя'] = properties['Значение показателя'].astype(str).str.replace(',', '.', regex=False).str.replace('°C','', regex=False)

# свойства уникальные для каждого ТИПА компонентов - собираем в словарь
# но не у каждого конкретного компонента полный набор свойств!
component_group_properties = {}
numeric_indicators = []
categorical_indicators = []

# заменяем 0.05(B) > 0.05
properties.loc[(properties["Наименование показателя"] == 'Содержание других элементов (например, бора)') & (properties["Значение показателя"] == '0.05 (B)'), "Значение показателя"] = 0.05

#
properties.loc[(properties["Наименование показателя"] == 'Щелочное число, ASTM D2896') & (properties["Значение показателя"] == '300-340'), "Значение показателя"] = 320
#
properties.loc[(properties["Наименование показателя"] == 'Содержание металла (Ca/Mg), % масс.') & (properties["Значение показателя"] == '9.0 (Mg)'), "Значение показателя"] = 9.0
#
properties.loc[(properties["Наименование показателя"] == 'Плотность при 15°С, ASTM D4052') & (properties["Значение показателя"] == '0.98-1.05'), "Значение показателя"] = 1.015
#

for group_name in properties['component_group'].unique():
    group_df = properties[properties['component_group'] == group_name]
    all_indicators = group_df['Наименование показателя'].dropna().unique()

    valid = [] # валидный - 1) нет NaN-свойств 2) если свойства не NaN, то показатели свойств не NaN и не 0
    invalid = []

    for indicator in all_indicators:
      values = group_df[group_df['Наименование показателя'] == indicator]['Значение показателя']

      if values.isna().all():
            invalid.append(indicator)
            continue

      numeric_vals = pd.to_numeric(values, errors='coerce')
      if not numeric_vals.isna().any() and (numeric_vals == 0).all():
          invalid.append(indicator)
          continue

      valid.append(indicator)

      if not numeric_vals.isna().any():
            numeric_indicators.append(indicator)
      else:
            categorical_indicators.append(indicator)

    component_group_properties[group_name] = set(valid)

    if len(valid) == 0:
      print(f"Group '{group_name}': 0 валидных свойств.")
    if len(invalid) > 0:
      print(f"Group '{group_name}': Импостеры с полностью NaN/zero значениями: {invalid}")

Group 'Антипенная_присадка': 0 валидных свойств.
Group 'Антипенная_присадка': Импостеры с полностью NaN/zero значениями: ['Массовая доля кальция, ASTM D6481', 'Массовая доля серы, ASTM D6481', 'Массовая доля фосфора, ASTM D6481', 'Массовая доля цинка, ASTM D6481']


In [11]:
# все, что превратилось в NaN когда применяли pd.to_numeric - исходно содерж строки-колонки либо исходно NaN - я почистила смешанные признаки (числовые с примесью строк)
set(categorical_indicators)
# размер мицелл - битый формат c примесью даты, не берем

{'CAS',
 'SMILES для наиболее вероятной (средней) молекулы сульфокислоты',
 'Категория',
 'Класс полиамина',
 'Класс субстрата',
 'Модификация',
 'Номер CAS',
 'Номер CAS / SMILES',
 'Происхождение',
 'Разветвленность радикала / радикалов',
 'Размер мицелл, нм',
 'Структура УВ-радикала',
 'Тип АО',
 'Тип лиганда',
 'Тип полимера',
 'Тип спиртового радикала',
 'Тип сукцинимида'}

In [12]:
non_numerical_columns = [
 'CAS',
 'SMILES для наиболее вероятной (средней) молекулы сульфокислоты',
 'Категория',
 'Класс полиамина',
 'Класс субстрата',
 'Модификация',
 'Номер CAS',
 'Номер CAS / SMILES',
 'Происхождение',
 'Разветвленность радикала / радикалов',
#'Размер мицелл, нм',
 'Структура УВ-радикала',
 'Тип АО',
 'Тип лиганда',
 'Тип полимера',
 'Тип спиртового радикала',
 'Тип сукцинимида']

numerical_feature_columns = list(set(numeric_indicators))

In [13]:
pivot_df = properties.pivot_table(
    index=['Компонент', 'Наименование партии'],
    columns='Наименование показателя',
    values='Значение показателя',
    aggfunc='first').reset_index()

pivot_df['component_group'] = pivot_df['Компонент'].apply(lambda x: re.sub(r'[_\d]+$', '', x))
prop_cols = [c for c in pivot_df.columns if c not in ['Компонент', 'Наименование партии', 'component_group']]
num_cols = [c for c in prop_cols if c in numerical_feature_columns]
cat_cols = [c for c in prop_cols if c in non_numerical_columns]

# заполнение из typical
for comp in pivot_df['Компонент'].unique():
    comp_mask = pivot_df['Компонент'] == comp

    # typical строка для этого компонента
    typical_rows = pivot_df[comp_mask & (pivot_df['Наименование партии'] == 'typical')]
    if typical_rows.empty:
        continue  # нет typical - ничего не заполняем

    typical_row = typical_rows.iloc[0]
    group_name = pivot_df.loc[comp_mask, 'component_group'].iloc[0]
    # достаем свойства для этой группы
    allowed_props = component_group_properties.get(group_name, set())

    for prop in prop_cols:
        if prop not in allowed_props:
            # свойство отсутствует для группы – его значения не трогаем
            continue

        # для всех партий компонента != typical заполняем NaN из typical, если typical значение не NaN
        typical_val = typical_row.get(prop)
        if pd.isna(typical_val):
            # В typical NaN - идем дальше
            continue
        # заполняем только те строки компонента, где текущее значение NaN и партия не typical
        fill_mask = (comp_mask & (pivot_df['Наименование партии'] != 'typical') & pivot_df[prop].isna())
        pivot_df.loc[fill_mask, prop] = typical_val


pivot_df = pivot_df.drop(columns=['component_group'])
pivot_df.set_index(['Компонент', 'Наименование партии'], inplace=True) # мультииндекс

pivot_df = pivot_df.dropna(axis=1, how='all')
display(pivot_df.head())

Наименование показателя             % масс. (Mo)  CAS COC (°C)  \
Компонент       Наименование партии                              
Антиоксидант_1  01028.23                     NaN  NaN      NaN   
                02717.22                     NaN  NaN      NaN   
                typical                      NaN  NaN      NaN   
Антиоксидант_10 4793.19                      NaN  NaN      NaN   
Антиоксидант_11 typical                      NaN  NaN      NaN   

Наименование показателя             SMILES для наиболее вероятной (средней) молекулы сульфокислоты  \
Компонент       Наименование партии                                                                  
Антиоксидант_1  01028.23                                                           NaN               
                02717.22                                                           NaN               
                typical                                                            NaN               
Антиоксидант_10 4793.19                                                            NaN               
Антиоксидант_11 typical                                                            NaN               

Наименование показателя             Активный Азот / Кислород, % масс. (N или O)  \
Компонент       Наименование партии                                               
Антиоксидант_1  01028.23                                            4.102564103   
                02717.22                                            4.102564103   
                typical                                             4.102564103   
Антиоксидант_10 4793.19                                                     NaN   
Антиоксидант_11 typical                                             4.102564103   

Наименование показателя             Анилиновая точка Атомное отношение P:Zn  \
Компонент       Наименование партии                                           
Антиоксидант_1  01028.23                         NaN                    NaN   
                02717.22                         NaN                    NaN   
                typical                          NaN                    NaN   
Антиоксидант_10 4793.19                          NaN                    NaN   
Антиоксидант_11 typical                          NaN                    NaN   

Наименование показателя             Группа по API Деаэрация | ASTM D3427  \
Компонент       Наименование партии                                        
Антиоксидант_1  01028.23                      NaN                    NaN   
                02717.22                      NaN                    NaN   
                typical                       NaN                    NaN   
Антиоксидант_10 4793.19                       NaN                    NaN   
Антиоксидант_11 typical                       NaN                    NaN   

Наименование показателя             Деэм.вода | ASTM D1401  ...  \
Компонент       Наименование партии                         ...   
Антиоксидант_1  01028.23                               NaN  ...   
                02717.22                               NaN  ...   
                typical                                NaN  ...   
Антиоксидант_10 4793.19                                NaN  ...   
Антиоксидант_11 typical                                NaN  ...   

Наименование показателя             Тип спиртового радикала Тип сукцинимида  \
Компонент       Наименование партии                                           
Антиоксидант_1  01028.23                                NaN             NaN   
                02717.22                                NaN             NaN   
                typical                                 NaN             NaN   
Антиоксидант_10 4793.19                                 NaN             NaN   
Антиоксидант_11 typical                                 NaN             NaN   

Наименование показателя             Химический потенциал, Дж/моль  \
Компонент       Наименование партии                         

In [14]:
num_columns = [col for col in num_cols if col not in ['Компонент', 'Наименование партии']]
num_features_df = pivot_df[num_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

In [15]:
typical_props = properties[properties['Наименование партии'] == 'typical'].copy()
typical_features_map = {}
for comp in typical_props['Компонент'].unique():
    comp_typical = typical_props[typical_props['Компонент'] == comp]
    if len(comp_typical) > 0:
        vec = []
        for col in num_columns:
            val = comp_typical.iloc[0].get(col)
            vec.append(float(val) if pd.notna(val) else 0.0)
        typical_features_map[comp] = np.array(vec, dtype=np.float32)

In [16]:
num_features_df.head()

Наименование показателя              % масс. (Mo)  COC (°C)  \
Компонент       Наименование партии                           
Антиоксидант_1  01028.23                      0.0       0.0   
                02717.22                      0.0       0.0   
                typical                       0.0       0.0   
Антиоксидант_10 4793.19                       0.0       0.0   
Антиоксидант_11 typical                       0.0       0.0   

Наименование показателя              Активный Азот / Кислород, % масс. (N или O)  \
Компонент       Наименование партии                                                
Антиоксидант_1  01028.23                                                4.102564   
                02717.22                                                4.102564   
                typical                                                 4.102564   
Антиоксидант_10 4793.19                                                 0.000000   
Антиоксидант_11 typical                                                 4.102564   

Наименование показателя              Анилиновая точка  Атомное отношение P:Zn  \
Компонент       Наименование партии                                             
Антиоксидант_1  01028.23                          0.0                     0.0   
                02717.22                          0.0                     0.0   
                typical                           0.0                     0.0   
Антиоксидант_10 4793.19                           0.0                     0.0   
Антиоксидант_11 typical                           0.0                     0.0   

Наименование показателя              Группа по API  Деаэрация | ASTM D3427  \
Компонент       Наименование партии                                          
Антиоксидант_1  01028.23                       0.0                     0.0   
                02717.22                       0.0                     0.0   
                typical                        0.0                     0.0   
Антиоксидант_10 4793.19                        0.0                     0.0   
Антиоксидант_11 typical                        0.0                     0.0   

Наименование показателя              Деэм.вода | ASTM D1401  \
Компонент       Наименование партии                           
Антиоксидант_1  01028.23                                0.0   
                02717.22                                0.0   
                typical                                 0.0   
Антиоксидант_10 4793.19                                 0.0   
Антиоксидант_11 typical                                 0.0   

Наименование показателя              Деэм.время | ASTM D1401  \
Компонент       Наименование партии                            
Антиоксидант_1  01028.23                                 0.0   
                02717.22                                 0.0   
                typical                                  0.0   
Антиоксидант_10 4793.19                                  0.0   
Антиоксидант_11 typical                                  0.0   

Наименование показателя              Деэм.масло | ASTM D1401  ...  \
Компонент       Наименование партии                           ...   
Антиоксидант_1  01028.23                                 0.0  ...   
                02717.22                                 0.0  ...   
                typical                                  0.0  ...   
Антиоксидант_10 4793.19                                  0.0  ...   
Антиоксидант_11 typical                                  0.0  ...   

Наименование показателя              Температура проведения теста | -  \
Компонент       Наименование партии                                     
Антиоксидант_1  01028.23                                          0.0   
                02717.22                                          0.0   
                typical                                           0.0   
Антиоксидант_10 4793.19                                           0.0   
Антиоксидант_11 typical                  

In [17]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(num_features_df)
X = features_scaled
sample_ids = pivot_df.index

In [18]:
print(f"Матрица числовых признаков: {X.shape}")

Матрица числовых признаков: (313, 77)


In [19]:
X

array([[-0.15033469, -0.14963193,  3.36575962, ..., -3.98328462,
         0.84988066,  3.64905879],
       [-0.15033469, -0.14963193,  3.36575962, ..., -3.98328462,
         0.84988066,  3.64905879],
       [-0.15033469, -0.14963193,  3.36575962, ..., -3.98328462,
         0.84988066,  3.64905879],
       ...,
       [ 7.03140476,  6.70424026, -0.24245425, ...,  0.26747665,
         0.19743198, -0.26813727],
       [ 7.03140476,  6.01885304, -0.24245425, ...,  0.26747665,
         0.19743198, -0.26813727],
       [ 7.03140476,  6.01885304, -0.24245425, ...,  0.26747665,
         0.19743198, -0.26813727]], shape=(313, 77))

In [ ]:
mixtures_train["Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C"].describe()

,"Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C"
count,2230.000000
mean,158.246637
std,3.760694
min,150.000000
25%,160.000000
50%,160.000000
75%,160.000000
max,160.000000


In [ ]:
mixtures_test["Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C"].describe()

,"Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C"
count,524.000000
mean,157.198473
std,4.428173
min,150.000000
25%,150.000000
50%,160.000000
75%,160.000000
max,160.000000


In [ ]:
mixtures_train[mixtures_train["scenario_id"] == "train_1"]

,scenario_id,Компонент,Наименование партии,"Массовая доля, %","Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C","Время испытания | - Daimler Oxidation Test (DOT), ч","Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %","Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm","Количество биотоплива | - Daimler Oxidation Test (DOT), % масс","Дозировка катализатора, категория"
0,train_1,Антиоксидант_5,32,70.832,160,168,25.4,98.04,0,1
1,train_1,Антиоксидант_6,13799.21,70.832,160,168,25.4,98.04,0,1
2,train_1,Антипенная_присадка_3,б/н,3.991,160,168,25.4,98.04,0,1
3,train_1,Базовое_масло_10,0285.21,60.034,160,168,25.4,98.04,0,1
4,train_1,Базовое_масло_17,без номера,84.281,160,168,25.4,98.04,0,1
5,train_1,Базовое_масло_2,0204.22,57.127,160,168,25.4,98.04,0,1
6,train_1,Базовое_масло_7,0186.21,99.414,160,168,25.4,98.04,0,1
7,train_1,Депрессорная_присадка_2,121256,17.074,160,168,25.4,98.04,0,1
8,train_1,Детергент_1,0174.22,93.897,160,168,25.4,98.04,0,1
9,train_1,Детергент_4,NaN,26.931,160,168,25.4,98.04,0,1


In [ ]:
mixtures_train[mixtures_train["Наименование партии"].isna()]

,scenario_id,Компонент,Наименование партии,"Массовая доля, %","Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C","Время испытания | - Daimler Oxidation Test (DOT), ч","Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %","Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm","Количество биотоплива | - Daimler Oxidation Test (DOT), % масс","Дозировка катализатора, категория"
9,train_1,Детергент_4,NaN,26.931,160,168,25.4,98.04,0,1
22,train_2,Детергент_4,NaN,26.931,160,168,20.5,112.37,0,1


In [ ]:
train_mixtures = mixtures_train

temp_vals = train_mixtures['Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C'].values.reshape(-1, 1)
time_vals = train_mixtures['Время испытания | - Daimler Oxidation Test (DOT), ч'].values.reshape(-1, 1)
biofuel_vals = train_mixtures['Количество биотоплива | - Daimler Oxidation Test (DOT), % масс'].values.reshape(-1, 1)
catalyst_raw = train_mixtures['Дозировка катализатора, категория'].astype(str).values

In [ ]:
scaler_temp = StandardScaler().fit(temp_vals)
scaler_time = StandardScaler().fit(time_vals)
scaler_biofuel = StandardScaler().fit(biofuel_vals)

In [ ]:
le_catalyst = LabelEncoder().fit(catalyst_raw)
# нормализуем для совпадения масштаба
catalyst_encoded = le_catalyst.transform(catalyst_raw).reshape(-1, 1)
scaler_catalyst = StandardScaler().fit(catalyst_encoded)

In [ ]:
all_targets = []
for sid in mixtures_train['scenario_id'].unique():
    row = mixtures_train[mixtures_train['scenario_id'] == sid].iloc[0]
    t1 = row['Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %']
    t2 = row['Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm']
    all_targets.append([t1, t2])

all_targets = np.array(all_targets)
scaler_y = StandardScaler()
scaler_y.fit(all_targets)
print(f"Target mean: {scaler_y.mean_}, std: {scaler_y.scale_}")

Target mean: [58.32353293 55.82754491], std: [196.42189814  36.00294849]
